In [0]:
import yaml
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, initcap, upper, to_date, row_number, current_timestamp, when, lit
)
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max
from delta.tables import DeltaTable

In [0]:
# SETUP & CONFIG
CONFIG_PATH = "/Workspace/Users/sarkaranurag2321@gmail.com/capstone-project/config/config.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["databricks"]["catalog"]
BRONZE_DB = config["databricks"]["schemas"]["bronze"]
SILVER_DB = config["databricks"]["schemas"]["silver"]
QUARANTINE_DB = config["databricks"]["schemas"]["quarantine"]

spark = SparkSession.builder.getOrCreate()

In [0]:
# Shared FUNCTIONS
def get_last_processed_ts(table_name):
    try:
        df = spark.table(table_name)
        return df.select(spark_max("ingestion_timestamp")).collect()[0][0]
    except:
        return None

def merge_to_silver(df, silver_table, primary_key, partition_col=None):
    if spark.catalog.tableExists(silver_table):
        target = DeltaTable.forName(spark, silver_table)
        target.alias("t").merge(
            df.alias("s"),
            f"t.{primary_key} = s.{primary_key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
    else:
        writer = df.write.format("delta").mode("overwrite")
        if partition_col:
            writer = writer.partitionBy(partition_col)
        writer.saveAsTable(silver_table)

def save_to_quarantine(df, table_base_name):
    """Saves invalid records to the dedicated QUARANTINE schema."""
    quarantine_table = f"{CATALOG}.{QUARANTINE_DB}.{table_base_name}_quarantine"
    count = df.count()
    if count > 0:
        print(f"Saving {count:,} invalid records to {quarantine_table}")
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(quarantine_table)
    else:
        print(f"No invalid data found for {table_base_name}.")

In [0]:

# Customers Table
print("\n--- Processing Customers ---")
c_bronze = config["datasets"]["customers"]["bronze_table"]
c_silver = config["datasets"]["customers"]["silver_table"]
c_pk = config["datasets"]["customers"]["primary_key"]

df = spark.table(f"{CATALOG}.{BRONZE_DB}.{c_bronze}")

last_ts = get_last_processed_ts(f"{CATALOG}.{SILVER_DB}.{c_silver}")
if last_ts:
    df = df.filter(col("ingestion_timestamp") > last_ts)

email_regex = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
df = df.withColumn("quarantine_reason", 
    when(col(c_pk).isNull(), "Missing PK")
    .when(~col("email").rlike(email_regex), "Invalid Email Format")
    .when(to_date(col("signup_date"), "yyyy-MM-dd").isNull(), "Invalid Date Format")
    .when(col("country") == "", "Missing Country")
    .otherwise(None)
)

invalid_df = df.filter(col("quarantine_reason").isNotNull())
valid_df = df.filter(col("quarantine_reason").isNull()).drop("quarantine_reason")

valid_df = (
    valid_df.withColumn("first_name", initcap(trim(col("first_name"))))
      .withColumn("last_name", initcap(trim(col("last_name"))))
      .withColumn("email", trim(col("email")))
      .withColumn("country", trim(col("country")))
      .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd"))
)

# Handling Deduplication
window = Window.partitionBy(c_pk).orderBy(col("ingestion_timestamp").desc())
valid_df = valid_df.withColumn("rn", row_number().over(window)).filter("rn = 1").drop("rn")

merge_to_silver(valid_df, f"{CATALOG}.{SILVER_DB}.{c_silver}", c_pk)
save_to_quarantine(invalid_df, "customers")

In [0]:
# Product Table

print("\n--- Processing Products ---")
p_bronze = config["datasets"]["products"]["bronze_table"]
p_silver = config["datasets"]["products"]["silver_table"]
p_pk = config["datasets"]["products"]["primary_key"]

df = spark.table(f"{CATALOG}.{BRONZE_DB}.{p_bronze}")

last_ts = get_last_processed_ts(f"{CATALOG}.{SILVER_DB}.{p_silver}")
if last_ts:
    df = df.filter(col("ingestion_timestamp") > last_ts)

df = df.withColumn("quarantine_reason", 
    when(col(p_pk).isNull(), "Missing PK")
    .when(col("price") <= 0, "Non-positive Price")
    .otherwise(None)
)

invalid_df = df.filter(col("quarantine_reason").isNotNull())
valid_df = df.filter(col("quarantine_reason").isNull()).drop("quarantine_reason")

valid_df = valid_df.withColumn("product_name", trim(col("product_name"))) \
                   .withColumn("category", initcap(trim(col("category"))))

merge_to_silver(valid_df, f"{CATALOG}.{SILVER_DB}.{p_silver}", p_pk)
save_to_quarantine(invalid_df, "products")

In [0]:
# Orders Table

print("\n--- Processing Orders ---")
o_bronze = config["datasets"]["orders"]["bronze_table"]
o_silver = config["datasets"]["orders"]["silver_table"]
o_pk = config["datasets"]["orders"]["primary_key"]
valid_statuses = config["datasets"]["orders"]["valid_statuses"]

df = spark.table(f"{CATALOG}.{BRONZE_DB}.{o_bronze}")

last_ts = get_last_processed_ts(f"{CATALOG}.{SILVER_DB}.{o_silver}")
if last_ts:
    df = df.filter(col("ingestion_timestamp") > last_ts)

df = df.withColumn("quarantine_reason", 
    when(col(o_pk).isNull(), "Missing PK")
    .when(~upper(trim(col("status"))).isin(valid_statuses), "Invalid Order Status")
    .when(to_date(col("order_date"), "yyyy-MM-dd").isNull(), "Invalid/Future Date")
    .otherwise(None)
)

invalid_df = df.filter(col("quarantine_reason").isNotNull())
valid_df = df.filter(col("quarantine_reason").isNull()).drop("quarantine_reason")

valid_df = valid_df.withColumn("status", upper(trim(col("status")))) \
                   .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

merge_to_silver(valid_df, f"{CATALOG}.{SILVER_DB}.{o_silver}", o_pk, config["gold"]["partition_column"])
save_to_quarantine(invalid_df, "orders")

In [0]:
# Order Items table

print("\n--- Processing Order Items ---")
oi_bronze = config["datasets"]["order_items"]["bronze_table"]
oi_silver = config["datasets"]["order_items"]["silver_table"]
oi_pk = config["datasets"]["order_items"]["primary_key"]

df = spark.table(f"{CATALOG}.{BRONZE_DB}.{oi_bronze}")

last_ts = get_last_processed_ts(f"{CATALOG}.{SILVER_DB}.{oi_silver}")
if last_ts:
    df = df.filter(col("ingestion_timestamp") > last_ts)

# Orphan check
silver_orders = spark.table(f"{CATALOG}.{SILVER_DB}.{o_silver}").select("order_id").withColumn("exists", lit(True))
df = df.join(silver_orders, on="order_id", how="left")

df = df.withColumn("quarantine_reason", 
    when(col(oi_pk).isNull(), "Missing PK")
    .when(col("quantity") <= 0, "Zero/Negative Quantity")
    .when(col("exists").isNull(), "Orphan Record (Order ID not in Silver)")
    .otherwise(None)
).drop("exists")

invalid_df = df.filter(col("quarantine_reason").isNotNull())
valid_df = df.filter(col("quarantine_reason").isNull()).drop("quarantine_reason")

merge_to_silver(valid_df, f"{CATALOG}.{SILVER_DB}.{oi_silver}", oi_pk)
save_to_quarantine(invalid_df, "order_items")

print(f"\nSILVER TRANSFORMATION — COMPLETE")